# 再現性の確保

---
## 目的
深層学習の実験における乱数シードの役割を理解し，`torch.manual_seed`や`torch.backends.cudnn.deterministic`などを用いて実験の再現性を確保する方法と，その限界について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 再現性とは

ネットワークの学習には，多くの箇所で乱数が使用されています．

- ネットワークの重みの初期値
- `DataLoader`で`shuffle=True`とした場合のデータの読み込み順序
- Data Augmentationにおけるランダムなクロップや反転などの処理
- Dropoutにより無効化されるユニットの選択

これらが実行のたびに異なると，全く同じコードを実行しても得られる結果（学習曲線や最終的な精度）が毎回変化してしまいます．
研究や開発においては，条件を変えたときの効果を正しく比較したり，不具合の原因を再現・特定したりするために，乱数を制御して同じ結果を再現できるようにしておくことが重要です．

## 乱数シードの固定

PyTorchでの学習には，Python標準の`random`，`numpy`，`torch`（CPU・GPU）といった複数のライブラリの乱数が関わっています．そのため，これらすべてに対してシードを設定する必要があります．
以下のように，1つの関数にまとめておくと便利です．

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    ### cuDNNが自動選択する計算アルゴリズムを固定し，毎回同じアルゴリズムを使用させる
    torch.backends.cudnn.deterministic = True
    ### 入力サイズに応じた最適なアルゴリズムを探索する機能（実行のたびに選択結果が変わりうる）を無効化する
    torch.backends.cudnn.benchmark = False

## 学習データとネットワークの準備

In [ ]:
train_data = torchvision.datasets.MNIST(root="./", train=True, transform=transforms.ToTensor(), download=True)
### 実行時間短縮のため，学習データの一部（先頭2000枚）のみを使用する
train_subset = torch.utils.data.Subset(train_data, range(2000))

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
        self.l1 = nn.Linear(14*14*4, 10)
        self.act = nn.Sigmoid()
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        h = self.pool(self.act(self.conv1(x)))
        h = h.view(h.size()[0], -1)
        h = self.l1(h)
        return h

## シードを固定した場合の再現性の確認

`set_seed()`を呼び出してから，ネットワークの作成・`DataLoader`の作成・学習を行う一連の処理を関数にまとめ，同じシードで2回実行して，得られるlossの推移が完全に一致するかを確認します．

In [ ]:
def run_training(seed, epoch_num=2):
    set_seed(seed)

    model = CNN()
    model = model.to(device)
    model.train()

    train_loader = torch.utils.data.DataLoader(train_subset, batch_size=100, shuffle=True)

    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    criterion = criterion.to(device)

    losses = []
    for epoch in range(epoch_num):
        for image, label in train_loader:
            image = image.to(device)
            label = label.to(device)

            y = model(image)
            loss = criterion(y, label)

            model.zero_grad()
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

    return losses

### 同じシード（seed=0）で2回学習を実行
losses_run1 = run_training(seed=0)
losses_run2 = run_training(seed=0)

print("1回目の先頭5イテレーションのloss:", [round(v, 6) for v in losses_run1[:5]])
print("2回目の先頭5イテレーションのloss:", [round(v, 6) for v in losses_run2[:5]])
print("2回の実行結果が完全に一致するか:", losses_run1 == losses_run2)

In [ ]:
### 異なるシード（seed=1）で学習を実行
losses_run3 = run_training(seed=1)

print("3回目（seed=1）の先頭5イテレーションのloss:", [round(v, 6) for v in losses_run3[:5]])
print("seed=0の実行結果と一致するか:", losses_run1 == losses_run3)

## 再現性確保の限界

シードを固定することで多くの場合は結果を再現できますが，以下のような限界があることに注意してください．

- **計算速度とのトレードオフ**：`torch.backends.cudnn.benchmark = False`は，入力サイズに応じた最適なアルゴリズムの自動探索を無効化するため，畳み込み演算の実行速度が低下する場合があります．学習速度を優先する場合は`True`のままにすることも多いですが，その場合は完全な再現性は得られません．
- **`DataLoader`のワーカープロセス**：`num_workers`を1以上に設定すると，データの読み込みが複数のプロセスで並列に行われます．各プロセスは独立した乱数状態を持つため，`worker_init_fn`引数などを使ってプロセスごとにシードを設定しない限り，データの読み込み順序やAugmentationの結果が実行のたびに変化する可能性があります．
- **環境依存の非決定性**：GPUの種類やCUDA・cuDNNのバージョンが異なる環境間では，浮動小数点演算の丸め誤差などにより，シードを固定しても完全に同一の結果が得られるとは限りません．
- **`torch.use_deterministic_algorithms(True)`**：PyTorchが提供する，決定的な実装を持たない演算の使用時にエラーを発生させるための機能です．より厳密な再現性を確保できますが，一部の演算が利用できなくなる場合があります．

このように，シード固定は再現性を大きく改善しますが，「常に完全に同一の結果が得られることを保証するもの」ではない点に留意してください．

## 課題

1. `set_seed()`内の`torch.backends.cudnn.deterministic`と`torch.backends.cudnn.benchmark`の設定を変更（`deterministic = False`, `benchmark = True`）し，同じシードで学習を2回実行した場合に，lossの推移が完全に一致するかどうかを確認してみましょう．

2. `run_training()`の学習データ数やエポック数を増やし，学習にかかる時間が`cudnn.benchmark`の設定によってどの程度変化するか比較してみましょう．